In [24]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
End-to-end EFA on WVS-style data with item map.

Inputs (same folder):
  - data-wvs.csv : the survey data
  - map-wvs.json : dict { "Q1": "full question text", ... }

What it does:
  1) Loads data and maps question IDs to text.
  2) Keeps only columns present in map (Q1..Q259, etc.).
  3) Missing-value filtering (≤50% missing per item/respondent).
  4) Mean imputation + z-score standardization.
  5) Horn's Parallel Analysis (200 sims) + Velicer's MAP.
  6) Chooses factor count (consensus + sanity cap) — default cap=15.
  7) FactorAnalysis + varimax rotation.
  8) Saves:
        - 'scree_parallel.png'
        - 'variance_by_factor.png'
        - 'wvs_rotated_loadings.csv'
  9) Prints a concise summary and top 10 loadings per factor.

Dependencies:
  python>=3.9, numpy, pandas, matplotlib, scikit-learn
"""

import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import eigh, eigvalsh, svd
from sklearn.decomposition import FactorAnalysis
from pathlib import Path
import re
import json
from collections import Counter, defaultdict

# ----------------------------
# Configuration
# ----------------------------
CSV_PATH = Path("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/wvs/data-wvs.csv")
MAP_PATH = Path("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/wvs/map-wvs.json")
PA_BOOTSTRAPS = 200         # parallel analysis replications
MAX_FACTORS_CAP = 15        # final cap for interpretability
RANDOM_STATE = 42           # reproducibility
TOP_N_PER_FACTOR = 10       # for console printout

# ----------------------------
# Helpers
# ----------------------------
def varimax(Phi, gamma=1.0, q=20, tol=1e-6):
    """
    Orthogonal varimax rotation (Kaiser, 1958).
    Phi: (p x k) loadings matrix
    Returns rotated loadings and rotation matrix.
    """
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for i in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = svd(
            Phi.T @ (Lambda**3 - (gamma/p) * Lambda @ np.diag(np.sum(Lambda**2, axis=0)))
        )
        R = u @ vh
        d = np.sum(s)
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ R, R


def parallel_analysis(n, p, B=200, rng=None):
    """
    Horn's Parallel Analysis for correlation matrices.
    Returns the 95th percentile of random eigenvalues (descending).
    """
    if rng is None:
        rng = np.random.default_rng(RANDOM_STATE)
    rand_eigs_all = np.zeros((B, p))
    for b in range(B):
        Z = rng.standard_normal(size=(n, p))
        Z = (Z - Z.mean(axis=0)) / Z.std(axis=0, ddof=0)
        Rb = np.corrcoef(Z, rowvar=False)
        eb = np.sort(eigvalsh(Rb))[::-1]
        rand_eigs_all[b, :] = eb
    return np.quantile(rand_eigs_all, 0.95, axis=0)


def velicer_map(R, vals, vecs):
    """
    Velicer's MAP test using squared partial correlations
    via principal components (O'Connor, 2000 approach).
    Returns k_map (argmin of average squared partials).
    """
    # Guard against small negative numerical eigenvalues
    vals_pos = np.clip(vals, 0, None)
    loadings_pca = vecs * np.sqrt(vals_pos)

    S = R.copy()
    np.fill_diagonal(S, 0)
    p = R.shape[0]
    mean_sq_partials = []
    for m in range(p):
        if m == 0:
            resid = S.copy()
        else:
            Lm = loadings_pca[:, :m]
            Rm = Lm @ Lm.T
            np.fill_diagonal(Rm, 0)
            resid = S - Rm
        off = resid[~np.eye(p, dtype=bool)]
        mean_sq_partials.append(float(np.nanmean(off**2)))
    return int(np.argmin(mean_sq_partials))


def choose_k(vals, pa_p95, k_map, cap=15):
    """
    Combine PA & MAP; use Kaiser as a tie-breaker; cap for interpretability.
    """
    k_parallel = int(np.sum(vals > pa_p95))
    k_kaiser = int(np.sum(vals > 1.0))

    if abs(k_parallel - k_map) <= 1:
        k = max(k_parallel, k_map)
    else:
        # pick the one closer to Kaiser
        k = min([k_parallel, k_map], key=lambda v: abs(v - k_kaiser))

    k = max(1, min(int(k), cap))
    return k, k_parallel, k_map, k_kaiser



def print_factor_summary(loadings_df, k, top_n=10):
    """
    Print top-N absolute loadings per factor, with question text.

    Fixes:
      - Use .iloc to select by position (not .reindex)
      - Coerce index to string for safe formatted printing
    """
    import numpy as np

    for j in range(1, k + 1):
        col = f"Factor{j}"
        # positions of top |loading| items
        top_idx = np.argsort(-np.abs(loadings_df[col].to_numpy()))[:top_n]
        sub = loadings_df.iloc[top_idx]

        print(f"\n=== Top {top_n} items for {col} ===")
        for item, row in sub.iterrows():
            # Coerce item to string to avoid 's' format error if index is int
            print(f"{str(item):>8}  {row[col]: .3f}  |  {row['Question']}")

def _normalize(text: str) -> str:
    """Basic, language-agnostic text cleaning for keyword matches."""
    text = text.lower()
    # keep letters and spaces
    text = re.sub(r"[^a-zà-ž0-9 \-\_']", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _tokenize(text: str):
    return _normalize(text).split()

def extract_top_items_per_factor(loadings_df: pd.DataFrame,
                                 k: int,
                                 top_n: int = 15,
                                 min_abs_loading: float = 0.35):
    """
    Return dict: factor -> list of tuples (item_id, abs_loading, question_text)
    based on absolute loadings descending, filtered by min_abs_loading.
    """
    result = {}
    for j in range(1, k + 1):
        col = f"Factor{j}"
        # top indices by absolute loading
        top_idx = np.argsort(-np.abs(loadings_df[col].to_numpy()))[:top_n]
        sub = loadings_df.iloc[top_idx][[col, "Question"]].copy()
        sub["abs_loading"] = sub[col].abs()
        sub = sub[sub["abs_loading"] >= min_abs_loading]
        items = [(idx, r["abs_loading"], r["Question"]) for idx, r in sub.iterrows()]
        result[col] = items
    return result

def default_keyword_lexicon():
    """
    A compact, editable lexicon mapping 'label' -> list of keywords/phrases.
    Matching is by substring on normalized question text.
    """
    return {
        # Morality / permissiveness
        "Personal morality & permissiveness": [
            "justifi", "sex before", "abortion", "divorce", "prostitution",
            "homosexual", "casual sex", "euthanasia"
        ],
        # Violence & rule-breaking
        "Rule-breaking & violence": [
            "steal", "bribe", "tax", "violence", "terrorism", "beat", "avoid fare",
            "claiming government benefits"
        ],
        # Religiosity & belief
        "Religiosity & beliefs": [
            "god", "religion", "pray", "religious services", "atheist", "heaven", "hell",
            "life after death", "religious faith"
        ],
        # Institutional confidence / democracy
        "Confidence in institutions (intl & national)": [
            "confidence", "united nations", "european union", "parliament",
            "imf", "world bank", "wto", "who", "icc", "nato", "civil service",
            "police", "elections"
        ],
        "Democracy & political system": [
            "democratic", "free elections", "rights", "civil rights", "equal rights",
            "satisfied with how the political system", "how democratically"
        ],
        # Social capital / membership
        "Associational membership & civic life": [
            "member", "organization", "union", "political party", "sport", "recreational",
            "environmental", "professional", "charitable", "women", "self-help"
        ],
        # Safety & crime
        "Neighborhood safety & victimization": [
            "crime", "unsafe", "robber", "violence in your neighborhood",
            "drug sale", "sexual harassment", "carrying much money",
            "not go out at night", "victim of a crime"
        ],
        # Immigration
        "Immigration threat vs benefit": [
            "immigration", "immigrants", "foreign workers", "terrorism", "unemployment",
            "social conflict", "cultural diversity", "asylum"
        ],
        # Subjective well-being & control
        "Subjective well-being & control": [
            "life satisfaction", "satisfied with the financial", "happy", "free choice",
            "control over their lives"
        ],
        # Information & media
        "Information sources & media": [
            "daily newspapers", "tv news", "radio news", "mobile phone",
            "email", "internet", "social media", "talk with friends"
        ],
        # Surveillance & privacy & generalized trust
        "Surveillance, privacy & generalized trust": [
            "video surveillance", "monitor all e-mails", "collect information",
            "most people can be trusted", "people you meet for the first time"
        ],
        # Place attachment / identity scale
        "Place attachment (local→global)": [
            "close to your village", "close to your town", "close to your city",
            "close to your county", "close to your region", "close to your country",
            "close to your continent", "close to the world"
        ],
        # Social distance / outgroup prejudice
        "Social distance toward out-groups": [
            "neighbors", "different race", "different religion", "different language",
            "immigrants", "unmarried couples living together"
        ],
        # Science & technology / modernity
        "Science & technology attitudes": [
            "science and technology", "opportunities for the next generation",
            "healthier, easier, comfortable", "world is better or worse off because of science",
        ],
        # Work ethic & responsibility (misc. public ethics)
        "Work ethic & public ethics": [
            "work is a duty", "work should always come first", "people who don't work turn lazy",
            "feeling of responsibility", "risk to be held accountable for giving or receiving a bribe",
        ],
    }

def score_factor_name(top_items, lexicon):
    """
    top_items: list[(item_id, abs_loading, question_text)]
    Return (best_label, match_details) where match_details has counts per label.
    """
    label_counts = Counter()
    q_texts = [q for _, _, q in top_items]
    joined = " || ".join(_normalize(q) for q in q_texts)

    # Count keyword hits per label
    for label, kws in lexicon.items():
        cnt = 0
        for kw in kws:
            kw_norm = _normalize(kw)
            if kw_norm and kw_norm in joined:
                cnt += joined.count(kw_norm)
        if cnt > 0:
            label_counts[label] = cnt

    if label_counts:
        best = label_counts.most_common(1)[0][0]
    else:
        best = None

    return best, dict(label_counts)

def auto_name_factors(loadings_df: pd.DataFrame,
                      k: int,
                      top_n: int = 15,
                      min_abs_loading: float = 0.35,
                      lexicon: dict | None = None,
                      overrides: dict | None = None,
                      rename_columns: bool = False):
    """
    Return dict of suggested names per factor; write names + diagnostics to files.
    If overrides provided, they replace suggestions for matching Factor{j}.
    If rename_columns=True, returns a copy of loadings_df with renamed factor columns.
    """
    if lexicon is None:
        lexicon = default_keyword_lexicon()

    topdict = extract_top_items_per_factor(loadings_df, k, top_n, min_abs_loading)

    suggested = {}
    debug_rows = []
    for j in range(1, k + 1):
        col = f"Factor{j}"
        best, details = score_factor_name(topdict[col], lexicon)
        name = best if best else f"Factor {j}"
        suggested[col] = name
        # Keep top 6 question snippets for inspection
        short_qs = [q[:90] for _, _, q in topdict[col][:6]]
        debug_rows.append({
            "Factor": col,
            "SuggestedName": name,
            "KeywordHitBreakdown": json.dumps(details, ensure_ascii=False),
            "TopExampleQuestions": " | ".join(short_qs),
        })

    # Apply overrides if any
    overrides = overrides or {}
    for fac, nm in overrides.items():
        if fac in suggested and isinstance(nm, str) and nm.strip():
            suggested[fac] = nm.strip()

    # Save names to disk
    with open("factor_names.json", "w", encoding="utf-8") as f:
        json.dump(suggested, f, ensure_ascii=False, indent=2)
    pd.DataFrame(debug_rows).to_csv("factor_names.csv", index=False)

    # Annotate items with primary factor (max |loading|)
    factor_cols = [f"Factor{j}" for j in range(1, k + 1)]
    abs_mat = loadings_df[factor_cols].abs().to_numpy()
    best_pos = abs_mat.argmax(axis=1)
    loadings_df["PrimaryFactor"] = [factor_cols[i] for i in best_pos]
    loadings_df["PrimaryLoadingAbs"] = abs_mat[np.arange(abs_mat.shape[0]), best_pos]
    loadings_df["PrimaryFactorName"] = [suggested[f] for f in loadings_df["PrimaryFactor"]]

    # Optionally rename factor columns
    if rename_columns:
        rename_map = {f: f"{f} — {suggested[f]}" for f in factor_cols}
        loadings_df_named = loadings_df.rename(columns=rename_map).copy()
    else:
        loadings_df_named = loadings_df.copy()

    loadings_df_named.to_csv("wvs_rotated_loadings_named.csv", index_label="Item")

    return suggested, loadings_df_named
# === End factor naming utilities ============================================

# ----------------------------
# Main
# ----------------------------
def main():
    # 1) Load data and question map
    wvs = pd.read_csv(CSV_PATH)
    with open(MAP_PATH, "r", encoding="utf-8") as f:
        qmap = json.load(f)

    # 2) Keep only mapped questionnaire items (Q1..Q259...)
    q_cols = [c for c in wvs.columns if c in qmap.keys()]
    X = wvs[q_cols].apply(pd.to_numeric, errors="coerce")

    # 3) Missingness filters: items and rows (≤50% missing)
    X = X.loc[:, X.isna().mean() <= 0.50]
    X = X.loc[X.isna().mean(axis=1) <= 0.50].copy()
    n, p = X.shape
    print(f"Kept N={n} respondents, p={p} items after missingness filters.")

    # 4) Mean imputation + z-score standardization
    X = X.fillna(X.mean())
    X = (X - X.mean()) / X.std(ddof=0)

    # 5) Correlation matrix + eigen decomposition
    R = np.corrcoef(X.values, rowvar=False)  # p x p
    vals, vecs = eigh(R)
    idx = np.argsort(vals)[::-1]
    vals = vals[idx]
    vecs = vecs[:, idx]

    # 6) Parallel Analysis threshold
    pa_p95 = parallel_analysis(n, p, B=PA_BOOTSTRAPS)

    # Scree plot with PA
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, p + 1), vals, "o-", label="Observed eigenvalues")
    plt.plot(range(1, p + 1), pa_p95, "r--", label="PA 95th percentile")
    plt.xlabel("Component number")
    plt.ylabel("Eigenvalue")
    plt.title("Scree plot with Parallel Analysis threshold")
    plt.legend()
    plt.tight_layout()
    plt.savefig("scree_parallel.png", dpi=200)
    plt.close()

    # 7) Velicer’s MAP
    k_map = velicer_map(R, vals, vecs)

    # 8) Choose factor count (capped)
    k_best, k_parallel, k_map2, k_kaiser = choose_k(vals, pa_p95, k_map, cap=MAX_FACTORS_CAP)

    print("\n--- Factor count selection ---")
    print(f"Parallel Analysis (95th pct): {k_parallel}")
    print(f"Velicer MAP:               : {k_map2}")
    print(f"Kaiser (>1.0):             : {k_kaiser}")
    print(f"Selected (capped @ {MAX_FACTORS_CAP}): {k_best}")

    # 9) Factor Analysis + varimax rotation
    fa = FactorAnalysis(n_components=k_best, random_state=RANDOM_STATE)
    fa.fit(X.values)
    loadings = fa.components_.T  # p x k
    rot_loadings, rot_matrix = varimax(loadings)

    # 10) Communalities / uniquenesses / variance share
    communalities = np.sum(rot_loadings**2, axis=1)
    uniquenesses = 1 - communalities
    factor_var = np.sum(rot_loadings**2, axis=0)
    prop_var = factor_var / np.sum(factor_var)

    # 11) Build loadings DataFrame with question text
    items = X.columns.tolist()
    q_texts = [qmap.get(c, c) for c in items]
    cols = [f"Factor{j+1}" for j in range(k_best)]
    loadings_df = pd.DataFrame(rot_loadings, index=items, columns=cols)
    loadings_df["Communality"] = communalities
    loadings_df["Uniqueness"]  = uniquenesses
    loadings_df["Question"]    = q_texts

    # 12) Save loadings matrix and plots
    loadings_df.to_csv("wvs_rotated_loadings.csv", index_label="Item")

    plt.figure(figsize=(8, 4))
    plt.bar([f"F{j+1}" for j in range(k_best)], prop_var)
    plt.ylabel("Proportion of common variance")
    plt.title("Variance explained by factor (rotated)")
    plt.tight_layout()
    plt.savefig("variance_by_factor.png", dpi=200)
    plt.close()

    # 13) Print summary + top items per factor
    print("\n--- Variance share by factor (rotated) ---")
    for j in range(k_best):
        print(f"Factor {j+1:>2d}: {prop_var[j]:.4f}")

    print_factor_summary(loadings_df, k_best, top_n=TOP_N_PER_FACTOR)

    print("\nArtifacts saved:")
    print("  - scree_parallel.png")
    print("  - variance_by_factor.png")
    print("  - wvs_rotated_loadings.csv")
    print("\nDone.")

    
    # After you have: loadings_df with columns Factor1..FactorK and 'Question'

    K = rot_loadings.shape[1]  # or your selected k_best

    # (optional) human-in-the-loop overrides to pin exact names if you like.
    # Uncomment and edit as you wish:
    # overrides = {
    #     "Factor1":  "Personal morality & permissiveness",
    #     "Factor2":  "Confidence in institutions (intl & national)",
    #     "Factor3":  "Associational membership & civic life",
    #     "Factor4":  "Rule-breaking & violence",
    #     "Factor5":  "Neighborhood safety & victimization",
    #     "Factor6":  "Immigration threat vs benefit",
    #     "Factor7":  "Religiosity & beliefs",
    #     "Factor8":  "Subjective well-being & control",
    #     "Factor9":  "Information sources & media",
    #     "Factor10": "Surveillance, privacy & generalized trust",
    #     "Factor11": "Place attachment (local→global)",
    #     "Factor12": "Social distance toward out-groups",
    #     "Factor13": "Work ethic & public ethics",
    #     "Factor14": "Science & technology attitudes",
    #     "Factor15": "Democracy & political system",
    # }

    overrides = None  # or set to the dict above
    names, loadings_df_named = auto_name_factors(
        loadings_df, k=K,
        top_n=15,                   # how many top items per factor to consider
        min_abs_loading=0.35,       # ignore tiny loadings for naming
        lexicon=None,               # use default keywords (or pass your own dict)
        overrides=overrides,        # plug-in your edits here
        rename_columns=True         # rename Factor columns with names in the CSV
    )

    print("\n--- Suggested factor names ---")
    for f, nm in names.items():
        print(f"{f}: {nm}")
    print("\nSaved:")
    print("  - factor_names.json")
    print("  - factor_names.csv")
    print("  - wvs_rotated_loadings_named.csv")

if __name__ == "__main__":
    main()


Kept N=66 respondents, p=259 items after missingness filters.

--- Factor count selection ---
Parallel Analysis (95th pct): 48
Velicer MAP:               : 65
Kaiser (>1.0):             : 39
Selected (capped @ 15): 15

--- Variance share by factor (rotated) ---
Factor  1: 0.3000
Factor  2: 0.1495
Factor  3: 0.0771
Factor  4: 0.0747
Factor  5: 0.0868
Factor  6: 0.0308
Factor  7: 0.0361
Factor  8: 0.0389
Factor  9: 0.0326
Factor 10: 0.0364
Factor 11: 0.0314
Factor 12: 0.0304
Factor 13: 0.0229
Factor 14: 0.0314
Factor 15: 0.0209

=== Top 10 items for Factor1 ===
    Q186   0.933  |  Please tell me for the following action whether you think it can always be justified, never be justified, or something in between: Sex before marriage.
    Q184   0.928  |  Please tell me for the following action whether you think it can always be justified, never be justified, or something in between: Abortion.
     Q38   0.921  |  How would you feel about the following statements? Do you agree or disagree? A

In [20]:

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# ===================== Clustering utilities =====================

def compute_factor_scores(fa: FactorAnalysis, Xstd: np.ndarray) -> np.ndarray:
    """
    Factor scores for each respondent (n x k) using sklearn FactorAnalysis.transform.
    Xstd should be the standardized item matrix used to fit FA (same columns/order).
    """
    scores = fa.transform(Xstd)  # shape: (n, k)
    # z-standardize factor scores across respondents to equalize scale before clustering
    scores = (scores - scores.mean(axis=0)) / scores.std(axis=0, ddof=0)
    return scores

def kmeans_silhouette_curve(Z: np.ndarray, k_min: int = 2, k_max: int = 10, random_state: int = 42):
    sil = {}
    inertia = {}
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init="auto", random_state=random_state)
        labels = km.fit_predict(Z)
        sil[k] = silhouette_score(Z, labels)
        inertia[k] = km.inertia_
    return sil, inertia

def gmm_bic_curve(Z: np.ndarray, k_min: int = 2, k_max: int = 10, random_state: int = 42, cov_type: str = "full"):
    bic = {}
    for k in range(k_min, k_max + 1):
        gmm = GaussianMixture(n_components=k, covariance_type=cov_type, random_state=random_state, n_init=5)
        gmm.fit(Z)
        bic[k] = gmm.bic(Z)
    return bic

def choose_k_from_curves(sil: dict, bic: dict, sil_floor: float = 0.20):
    """
    Combine KMeans silhouette (maximize) and GMM BIC (minimize).
    - If the two suggested K are within 1, take the larger.
    - Else, take silhouette K if silhouette >= sil_floor, otherwise the BIC K.
    """
    k_sil = max(sil, key=lambda k: sil[k])
    k_bic = min(bic, key=lambda k: bic[k])

    if abs(k_sil - k_bic) <= 1:
        k_final = max(k_sil, k_bic)
        rule = "Consensus (±1) → larger K"
    else:
        if sil[k_sil] >= sil_floor:
            k_final = k_sil
            rule = f"Silhouette >= {sil_floor:.2f} → KMeans choice"
        else:
            k_final = k_bic
            rule = "Low silhouette → BIC choice (GMM)"

    return k_final, k_sil, k_bic, rule

def fit_best_cluster_model(Z: np.ndarray, k: int, random_state: int = 42):
    """
    Fit both KMeans and GMM at the chosen K, return the one with higher silhouette.
    """
    km = KMeans(n_clusters=k, n_init="auto", random_state=random_state).fit(Z)
    gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=random_state, n_init=5).fit(Z)

    km_labels  = km.predict(Z)
    gmm_labels = gmm.predict(Z)

    km_sil  = silhouette_score(Z, km_labels)
    gmm_sil = silhouette_score(Z, gmm_labels)

    if gmm_sil > km_sil:
        return ("GMM", gmm, gmm_labels, gmm_sil, km_sil, None, None)
    else:
        return ("KMeans", km, km_labels, km_sil, gmm_sil, km.cluster_centers_, None)

def save_cluster_diagnostics(Z: np.ndarray, sil: dict, bic: dict, labels: np.ndarray, out_prefix: str = ""):
    # Silhouette curve
    plt.figure(figsize=(6, 4))
    ks = sorted(sil.keys())
    plt.plot(ks, [sil[k] for k in ks], "o-", label="Silhouette (KMeans)")
    plt.axhline(0.20, color="gray", ls="--", lw=1, label="Silhouette floor 0.20")
    plt.xlabel("K")
    plt.ylabel("Silhouette")
    plt.title("Silhouette vs. K")
    plt.legend()
    plt.tight_layout()
    plt.savefig("silhouette_by_k.png", dpi=200)
    plt.close()

    # BIC curve
    plt.figure(figsize=(6, 4))
    ks = sorted(bic.keys())
    plt.plot(ks, [bic[k] for k in ks], "o-", color="tab:orange", label="BIC (GMM, lower=better)")
    plt.xlabel("K")
    plt.ylabel("BIC")
    plt.title("BIC vs. K (GMM)")
    plt.legend()


In [ ]:
# ----------------------------
# Main Workflow: Factor Analysis & Clustering
# ----------------------------
def main():
    # 1) Load data and question map
    wvs = pd.read_csv(CSV_PATH)
    with open(MAP_PATH, "r", encoding="utf-8") as f:
        qmap = json.load(f)

    # 2) Keep only mapped questionnaire items (Q1..Q259...)
    q_cols = [c for c in wvs.columns if c in qmap.keys()]
    X = wvs[q_cols].apply(pd.to_numeric, errors="coerce")

    # 3) Missingness filters: items and rows (≤50% missing)
    X = X.loc[:, X.isna().mean() <= 0.50]
    X = X.loc[X.isna().mean(axis=1) <= 0.50].copy()
    n, p = X.shape
    print(f"Kept N={n} respondents, p={p} items after missingness filters.")

    # 4) Mean imputation + z-score standardization
    X = X.fillna(X.mean())
    X_std = (X - X.mean()) / X.std(ddof=0)

    # 5) Correlation matrix + eigen decomposition
    R = np.corrcoef(X_std.values, rowvar=False)  # p x p
    vals, vecs = eigh(R)
    idx = np.argsort(vals)[::-1]
    vals = vals[idx]
    vecs = vecs[:, idx]

    # 6) Parallel Analysis threshold
    pa_p95 = parallel_analysis(n, p, B=PA_BOOTSTRAPS)

    # Scree plot with PA
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, p + 1), vals, "o-", label="Observed eigenvalues")
    plt.plot(range(1, p + 1), pa_p95, "r--", label="PA 95th percentile")
    plt.xlabel("Component number")
    plt.ylabel("Eigenvalue")
    plt.title("Scree plot with Parallel Analysis threshold")
    plt.legend()
    plt.tight_layout()
    plt.savefig("scree_parallel.png", dpi=200)
    plt.close()

    # 7) Velicer’s MAP
    k_map = velicer_map(R, vals, vecs)

    # 8) Choose factor count (capped)
    k_best, k_parallel, k_map2, k_kaiser = choose_k(vals, pa_p95, k_map, cap=MAX_FACTORS_CAP)

    print("\n--- Factor count selection ---")
    print(f"Parallel Analysis (95th pct): {k_parallel}")
    print(f"Velicer MAP:                : {k_map2}")
    print(f"Kaiser (>1.0):              : {k_kaiser}")
    print(f"Selected (capped @ {MAX_FACTORS_CAP}): {k_best}")

    # 9) Factor Analysis + varimax rotation
    fa = FactorAnalysis(n_components=k_best, random_state=RANDOM_STATE)
    fa.fit(X_std.values)
    loadings = fa.components_.T  # p x k
    rot_loadings, rot_matrix = varimax(loadings)

    # 10) Communalities / uniquenesses / variance share
    communalities = np.sum(rot_loadings**2, axis=1)
    uniquenesses = 1 - communalities
    factor_var = np.sum(rot_loadings**2, axis=0)
    prop_var = factor_var / np.sum(factor_var)

    # 11) Build loadings DataFrame with question text
    items = X_std.columns.tolist()
    q_texts = [qmap.get(c, c) for c in items]
    fac_cols_base = [f"Factor{j+1}" for j in range(k_best)]
    loadings_df = pd.DataFrame(rot_loadings, index=items, columns=fac_cols_base)
    loadings_df["Communality"] = communalities
    loadings_df["Uniqueness"]  = uniquenesses
    loadings_df["Question"]    = q_texts

    # 12) Save loadings matrix and plots
    loadings_df.to_csv("wvs_rotated_loadings.csv", index_label="Item")

    plt.figure(figsize=(8, 4))
    plt.bar([f"F{j+1}" for j in range(k_best)], prop_var)
    plt.ylabel("Proportion of common variance")
    plt.title("Variance explained by factor (rotated)")
    plt.tight_layout()
    plt.savefig("variance_by_factor.png", dpi=200)
    plt.close()

    # 13) Automated Naming
    overrides = None  # Plug in your factor name dictionary here if desired
    names, loadings_df_named = auto_name_factors(
        loadings_df, k=k_best,
        top_n=15,
        min_abs_loading=0.35,
        lexicon=None,
        overrides=overrides,
        rename_columns=True
    )

    print("\n--- Suggested factor names ---")
    for f, nm in names.items():
        print(f"{f}: {nm}")

    print_factor_summary(loadings_df, k_best, top_n=TOP_N_PER_FACTOR)

    # 14) Factor scores for clustering
    # X_std is the standardized item matrix
    factor_scores = compute_factor_scores(fa, X_std.values)  # shape (n, k_best)

    # Explore Cluster K in a modest range
    k_min, k_max = 2, min(10, max(3, k_best))
    sil, inertia = kmeans_silhouette_curve(factor_scores, k_min=k_min, k_max=k_max, random_state=RANDOM_STATE)
    bic = gmm_bic_curve(factor_scores, k_min=k_min, k_max=k_max, random_state=RANDOM_STATE, cov_type="full")

    k_final, k_sil, k_bic, rule = choose_k_from_curves(sil, bic, sil_floor=0.20)
    
    print(f"\n--- Cluster K selection ---")
    print(f"K (silhouette argmax, KMeans): {k_sil}  | best silhouette = {sil[k_sil]:.3f}")
    print(f"K (BIC argmin, GMM)         : {k_bic}  | best BIC        = {bic[k_bic]:.0f}")
    print(f"Chosen K                    : {k_final} | rule: {rule}")

    # 15) Fit final cluster model
    model_name, model, labels, model_sil, alt_sil, km_centers, _ = fit_best_cluster_model(
        factor_scores, k=k_final, random_state=RANDOM_STATE
    )
    print(f"\nCluster model picked: {model_name} (silhouette = {model_sil:.3f})")

    # 16) Export & Save Results
    save_cluster_diagnostics(factor_scores, sil, bic, labels)

    fac_cols_short = [f"F{j+1}" for j in range(k_best)]
    scores_df = pd.DataFrame(factor_scores, columns=fac_cols_short)
    scores_df["cluster"] = labels
    scores_df.to_csv("factor_scores_with_clusters.csv", index=False)

    # Export cluster profiles
    if model_name == "KMeans":
        centers = pd.DataFrame(model.cluster_centers_, columns=fac_cols_short)
    else:
        centers = pd.DataFrame(model.means_, columns=fac_cols_short)
    
    centers["cluster"] = np.arange(len(centers))
    centers = centers[["cluster"] + fac_cols_short]
    centers.to_csv("cluster_profiles.csv", index=False)

    # Print summary
    sizes = pd.Series(labels).value_counts().sort_index()
    print("\n--- Cluster sizes ---")
    for c, n_c in sizes.items():
        print(f"Cluster {c}: n={n_c}")

    print("\nAll Artifacts saved successfully.")

if __name__ == "__main__":
    main()

In [23]:

# ----------------------------
# Cluster naming + descriptions (3 sentences each)
# ----------------------------
import json
import numpy as np
import pandas as pd

def _top_factors_by_direction(center_row: pd.Series,
                              label_map: dict,
                              top_k_pos: int = 3,
                              top_k_neg: int = 3,
                              pos_thr: float = 0.35,
                              neg_thr: float = -0.35):
    """
    For one cluster centroid (z-scaled factor scores), return:
      - top_pos: list of (label, score) for strongest positive factors
      - top_neg: list of (label, score) for strongest negative factors
    Thresholds keep only meaningful deviations; if none pass thresholds, we still
    return the top_k by magnitude to ensure something is reported.
    """
    # Keep only F# columns in the right order; map to readable names
    fac_cols = [c for c in center_row.index if c.startswith("F")]
    scores = center_row[fac_cols].to_dict()

    # Positive and negative candidates (apply thresholds)
    pos = sorted([(label_map[c], v) for c, v in scores.items() if v >= pos_thr],
                 key=lambda x: x[1], reverse=True)[:top_k_pos]
    neg = sorted([(label_map[c], v) for c, v in scores.items() if v <= neg_thr],
                 key=lambda x: x[1])[:top_k_neg]

    # Fallbacks if nothing crosses the thresholds: take top |z| magnitudes
    if len(pos) == 0:
        pos = sorted([(label_map[c], v) for c, v in scores.items()],
                     key=lambda x: abs(x[1]), reverse=True)
        pos = [p for p in pos if p[1] > 0][:top_k_pos]
    if len(neg) == 0:
        neg = sorted([(label_map[c], v) for c, v in scores.items()],
                     key=lambda x: abs(x[1]), reverse=True)
        neg = [n for n in neg if n[1] < 0][:top_k_neg]

    return pos, neg


def _format_list_for_text(items, decimals=2):
    """
    Turn [(label, value), ...] into 'Label₁ (+0.85), Label₂ (+0.52)'.
    """
    def fmt(sign):
        return f"{sign:+.{decimals}f}"
    return ", ".join([f"{lab} ({fmt(val)})" for lab, val in items])


def name_and_describe_clusters(centers: pd.DataFrame,
                               names: dict,
                               sizes: pd.Series,
                               k_best: int,
                               pos_thr: float = 0.35,
                               neg_thr: float = -0.35,
                               top_k_pos: int = 3,
                               top_k_neg: int = 3,
                               decimals: int = 2,
                               total_n: int | None = None):
    """
    Create human-readable names + 3-sentence descriptions for each cluster.

    Parameters
    ----------
    centers : DataFrame
        Cluster centroids (KMeans) or component means (GMM) in z-scaled factor space.
        Must contain columns 'F1'..'Fk' and a 'cluster' id column.
    names : dict
        Mapping 'Factor1'..'FactorK' -> readable factor names (from factor_names.json).
    sizes : Series
        Value counts of labels (index = cluster id, values = size).
    k_best : int
        Number of factors used.
    pos_thr, neg_thr : float
        Thresholds to consider positive/negative deviations meaningful.
    top_k_pos, top_k_neg : int
        Max number of positives/negatives to list per cluster.
    decimals : int
        Rounding for z values in text.
    total_n : int | None
        If provided, will include percent of total in overview sentence.

    Returns
    -------
    DataFrame with columns: cluster, name, overview, strengths, weaknesses, description_3_sentences
    Also saves CSV and JSON to disk.
    """
    # Map F1..Fk to factor readable names (fall back to 'Factor j' if not found)
    label_map = {f"F{j+1}": names.get(f"Factor{j+1}", f"Factor {j+1}") for j in range(k_best)}

    rows = []
    for _, row in centers.sort_values("cluster").iterrows():
        cid = int(row["cluster"])

        # Size / proportion
        n_c = int(sizes.loc[cid]) if cid in sizes.index else None
        pct = (100.0 * n_c / total_n) if (total_n and n_c is not None and total_n > 0) else None

        # Find distinctive highs / lows
        top_pos, top_neg = _top_factors_by_direction(
            row, label_map,
            top_k_pos=top_k_pos, top_k_neg=top_k_neg,
            pos_thr=pos_thr, neg_thr=neg_thr
        )

        # Generate a short cluster name like "High X; Low Y"
        name_pos = top_pos[0][0] if len(top_pos) else None
        name_neg = top_neg[0][0] if len(top_neg) else None
        if name_pos and name_neg:
            cluster_name = f"High {name_pos}; Low {name_neg}"
        elif name_pos:
            cluster_name = f"High {name_pos}"
        elif name_neg:
            cluster_name = f"Low {name_neg}"
        else:
            # fallback: pick the factor with largest |z|
            fac_cols = [c for c in row.index if c.startswith("F")]
            best = max(fac_cols, key=lambda c: abs(row[c]))
            cluster_name = f"Distinctive {label_map[best]}"

        # Build the three sentences
        pos_text = _format_list_for_text(top_pos, decimals=decimals) if len(top_pos) else "no clear positive distinctives"
        neg_text = _format_list_for_text(top_neg, decimals=decimals) if len(top_neg) else "no clear negative distinctives"

        if pct is not None:
            overview = (f"This cluster groups respondents with relatively high scores on {pos_text} and low scores on "
                        f"{neg_text}. It contains n={n_c} participants ({pct:.1f}% of the sample).")
        else:
            overview = (f"This cluster groups respondents with relatively high scores on {pos_text} and low scores on "
                        f"{neg_text}. It contains n={n_c} participants.")

        if len(top_pos):
            strengths = f"Strengths include pronounced elevation on {pos_text}, indicating these tendencies stand out within this group."
        else:
            strengths = "Strengths: no clear positive factor elevations emerged beyond random variation."

        if len(top_neg):
            weaknesses = f"Weaknesses include pronounced reduction on {neg_text}, suggesting comparatively lower emphasis on these dimensions."
        else:
            weaknesses = "Weaknesses: no clear negative factor reductions emerged beyond random variation."

        description_3 = f"{overview} {strengths} {weaknesses}"

        rows.append({
            "cluster": cid,
            "name": cluster_name,
            "overview": overview,
            "strengths": strengths,
            "weaknesses": weaknesses,
            "description_3_sentences": description_3
        })

    out_df = pd.DataFrame(rows).sort_values("cluster").reset_index(drop=True)
    out_df.to_csv("cluster_names_and_descriptions.csv", index=False)
    with open("cluster_names_and_descriptions.json", "w", encoding="utf-8") as f:
        json.dump(out_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

    # Console preview
    print("\n--- Cluster names & descriptions ---")
    for _, r in out_df.iterrows():
        print(f"\nCluster {int(r['cluster'])} — {r['name']}")
        print(r["description_3_sentences"])

    print("\nSaved:")
    print("  - cluster_names_and_descriptions.csv")
    print("  - cluster_names_and_descriptions.json")

    return out_df
    
# If running standalone after your previous script completed:
centers = pd.read_csv("cluster_profiles.csv")
scores_df = pd.read_csv("factor_scores_with_clusters.csv")
with open("factor_names.json", "r", encoding="utf-8") as f:
    names = json.load(f)

# Recreate sizes and k_best from files (k_best = number of factor columns)
sizes = scores_df["cluster"].value_counts().sort_index()
k_best = len([c for c in scores_df.columns if c.startswith("F")])

_ = name_and_describe_clusters(
    centers=centers,
    names=names,
    sizes=sizes,
    k_best=k_best,
    total_n=len(scores_df)
)


--- Cluster names & descriptions ---

Cluster 0 — High Personal morality & permissiveness; Low Neighborhood safety & victimization
This cluster groups respondents with relatively high scores on Personal morality & permissiveness (+0.61) and low scores on Neighborhood safety & victimization (-1.40), Confidence in institutions (intl & national) (-1.13), Personal morality & permissiveness (-1.02). It contains n=8 participants (12.1% of the sample). Strengths include pronounced elevation on Personal morality & permissiveness (+0.61), indicating these tendencies stand out within this group. Weaknesses include pronounced reduction on Neighborhood safety & victimization (-1.40), Confidence in institutions (intl & national) (-1.13), Personal morality & permissiveness (-1.02), suggesting comparatively lower emphasis on these dimensions.

Cluster 1 — High Information sources & media; Low Religiosity & beliefs
This cluster groups respondents with relatively high scores on Information sources & m

In [14]:

# -*- coding: utf-8 -*-
"""
Country summary from EFA factor scores (example: Argentina).

What this script does
---------------------
1) Load the WVS-style dataset and the question map (Q1..Q259 → full text).
2) Keep only Q* columns present in the map; filter missingness (≤ 50% per item/respondent).
3) Mean-impute & z-standardize items.
4) Fit FactorAnalysis (k=15) and compute factor scores; z-standardize factor scores.
5) Varimax-rotate loadings (for interpretive labels).
6) Compute country-level mean factor scores minus global mean and list top +/- deviations.
7) Print a concise 3-sentence summary for the requested country.

Note
----
- With many items and relatively few respondents, this is exploratory.
- Factor signs are arbitrary; we compare *relative* deviations vs. the global mean.
"""

import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import FactorAnalysis


# ---------------------------
# Config
# ---------------------------

K_FACTORS = 15
COUNTRY_NAME = "Argentina"  # change to any country present in B_COUNTRY_ALPHA
ITEM_MISSING_MAX = 0.50     # ≤ 50% missing per item
ROW_MISSING_MAX = 0.50      # ≤ 50% missing per respondent


# ---------------------------
# Utilities
# ---------------------------
def varimax(Phi, gamma=1.0, q=20, tol=1e-6):
    """
    Orthogonal varimax rotation (Kaiser, 1958).
    Phi: (p x k) loading matrix
    Returns (rotated_loadings, rotation_matrix).
    """
    p, k = Phi.shape
    R = np.eye(k)
    d = 0.0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        # The expression below is the usual varimax update
        U, S, Vt = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p) * Lambda @ np.diag(np.sum(Lambda**2, axis=0)))
        )
        R = U @ Vt
        d = np.sum(S)
        if d_old != 0 and d / d_old < 1 + tol:
            break
    return Phi @ R, R


def prepare_data(csv_path: Path, map_path: Path):
    """Load data and return (X_std, qmap, meta) after filtering & standardization."""
    wvs = pd.read_csv(csv_path)
    with open(map_path, "r", encoding="utf-8") as f:
        qmap = json.load(f)

    # Keep only mapped Q* columns
    q_cols = [c for c in wvs.columns if c in qmap.keys()]
    X = wvs[q_cols].apply(pd.to_numeric, errors="coerce")
    meta = wvs[["B_COUNTRY_ALPHA"]].copy()

    # Filter items with ≤ 50% missing
    X = X.loc[:, X.isna().mean() <= ITEM_MISSING_MAX]
    # Filter respondents with ≤ 50% missing across retained items
    valid_rows = X.isna().mean(axis=1) <= ROW_MISSING_MAX
    X = X.loc[valid_rows].copy()
    meta = meta.loc[valid_rows].reset_index(drop=True)

    # Mean-impute then z-score standardize items
    X = X.fillna(X.mean())
    X_std = (X - X.mean()) / X.std(ddof=0)

    return X_std, qmap, meta


def fit_fa_and_scores(X_std: pd.DataFrame, k: int, random_state: int = 42):
    """Fit FA, return (fa_model, scores_z, loadings)."""
    fa = FactorAnalysis(n_components=k, random_state=random_state)
    fa.fit(X_std.values)
    # factor scores for respondents
    scores = fa.transform(X_std.values)
    # z-standardize the factor scores across respondents (per factor)
    scores_z = (scores - scores.mean(axis=0)) / scores.std(axis=0, ddof=0)
    # unrotated loadings: components_.T shape (p, k)
    loadings = fa.components_.T
    return fa, scores_z, loadings


def factor_labels_from_loadings(rot_loadings: np.ndarray, items, qmap, topn: int = 1):
    """
    Quick heuristic labels per factor:
    - take the top-|loading| items and pull a short snippet from the first question text.
    """
    labels = {}
    L = pd.DataFrame(rot_loadings, index=items,
                     columns=[f"F{j+1}" for j in range(rot_loadings.shape[1])])
    for j in range(rot_loadings.shape[1]):
        col = f"F{j+1}"
        top_idx = np.argsort(-np.abs(L[col].values))[:topn]
        # use first top item text as shorthand
        q_texts = [qmap.get(items[i], items[i]) for i in top_idx]
        # clean to a short phrase
        label = q_texts[0].split("?")[0][:60] if q_texts else f"Factor {j+1}"
        labels[col] = label
    return labels


def summarize_country(scores_df: pd.DataFrame,
                      country_col: str,
                      country_name: str,
                      labels: dict):
    """
    Compute country mean factor scores - global mean (all in z-units),
    list top 3 positive and top 3 negative deviations,
    and build a concise 3-sentence textual summary.
    """
    fac_cols = [c for c in scores_df.columns if c.startswith("F")]
    global_mean = scores_df[fac_cols].mean()

    subset = scores_df.loc[scores_df[country_col] == country_name]
    n_country = subset.shape[0]
    if n_country == 0:
        raise ValueError(f"No respondents found for '{country_name}' after filtering.")

    country_mean = subset[fac_cols].mean()
    diff = (country_mean - global_mean).sort_values(ascending=False)

    top_pos = diff.head(3)
    top_neg = diff.tail(3)

    # Human-readable strings with labels
    def fmt_triples(s):
        items = []
        for f, val in s.items():
            name = labels.get(f, f)
            items.append(f"{name} ({val:+.2f})")
        return ", ".join(items)

    pos_str = fmt_triples(top_pos)
    neg_str = fmt_triples(top_neg)

    # Three-sentence summary
    strengths = (f"In this sample, {country_name} stands out for {pos_str} "
                 f"(largest positive deviations vs. the global mean).")
    weaknesses = (f"Conversely, it is relatively lower on {fmt_triples(top_neg)} "
                  f"compared with the overall average.")
    bottom_line = (f"Overall, {country_name} shows distinctive strengths alongside "
                   f"areas of comparatively lower emphasis across the latent factors.")

    return {
        "n_country": int(n_country),
        "top_positive": top_pos.to_dict(),
        "top_negative": top_neg.to_dict(),
        "strengths_sentence": strengths,
        "weaknesses_sentence": weaknesses,
        "conclusion_sentence": bottom_line
    }


# ---------------------------
# Run
# ---------------------------
if __name__ == "__main__":
    # 1) Prepare matrix
    X_std, qmap, meta = prepare_data(CSV_PATH, MAP_PATH)
    print(f"Data ready: N={X_std.shape[0]} respondents, p={X_std.shape[1]} items.")

    # 2) Fit FA and compute factor scores
    fa, scores_z, loadings = fit_fa_and_scores(X_std, k=K_FACTORS, random_state=42)

    # 3) Rotate loadings and create quick labels
    rot_loadings, _ = varimax(loadings)
    items = X_std.columns.tolist()
    labels = factor_labels_from_loadings(rot_loadings, items, qmap, topn=1)

    # 4) Assemble factor-score dataframe with country
    scores_df = pd.DataFrame(scores_z, columns=[f"F{i+1}" for i in range(K_FACTORS)])
    scores_df["country"] = meta["B_COUNTRY_ALPHA"].values

    # 5) Summarize the chosen country
    result = summarize_country(scores_df, country_col="country",
                               country_name=COUNTRY_NAME, labels=labels)

    # 6) Print a concise report
    print("\n=== COUNTRY FACTOR SUMMARY ===")
    print(f"Country: {COUNTRY_NAME}  |  n={result['n_country']}")
    print("Top positive deviations:", result["top_positive"])
    print("Top negative deviations:", result["top_negative"])

    print("\n--- Three-sentence summary ---")
    print(result["strengths_sentence"])
    print(result["weaknesses_sentence"])
    print(result["conclusion_sentence"])


Data ready: N=66 respondents, p=259 items.

=== COUNTRY FACTOR SUMMARY ===
Country: Argentina  |  n=1
Top positive deviations: {'F13': 1.8665077541808643, 'F14': 1.2771137014077822, 'F12': 0.7723510179434153}
Top negative deviations: {'F10': -0.8082845675238773, 'F2': -1.3296338806332377, 'F6': -1.512533965025414}

--- Three-sentence summary ---
In this sample, Argentina stands out for How high is the risk in your country to be held accountable  (+1.87), Would you say the world is better or worse off because of sc (+1.28), On this list are various groups of people. Could you please  (+0.77) (largest positive deviations vs. the global mean).
Conversely, it is relatively lower on Do you agree that we depend too much on science and not enou (-0.81), How much confidence do you have in the following organizatio (-1.33), Do you agree that immigration increases the crime rate (-1.51) compared with the overall average.
Overall, Argentina shows distinctive strengths alongside areas of comparati

In [16]:
result["strengths_sentence"]

'In this sample, Argentina stands out for How high is the risk in your country to be held accountable  (+1.87), Would you say the world is better or worse off because of sc (+1.28), On this list are various groups of people. Could you please  (+0.77) (largest positive deviations vs. the global mean).'

In [17]:
result["weaknesses_sentence"]

'Conversely, it is relatively lower on Do you agree that we depend too much on science and not enou (-0.81), How much confidence do you have in the following organizatio (-1.33), Do you agree that immigration increases the crime rate (-1.51) compared with the overall average.'

In [18]:
result["conclusion_sentence"]

'Overall, Argentina shows distinctive strengths alongside areas of comparatively lower emphasis across the latent factors.'